In [1]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras import regularizers
REG = regularizers.l2(1e-4)

# ── 1. CUSTOM LAYERS (Bypassing Keras Save Bugs) ───────────────────────
class TileHorizon(layers.Layer):
    """Expands a single 2D spatial map into multiple identical future frames."""
    def __init__(self, horizon, **kwargs):
        super().__init__(**kwargs)
        self.horizon = horizon
    def call(self, x):
        return tf.tile(tf.expand_dims(x, 1), [1, self.horizon, 1, 1, 1])
    def get_config(self):
        return {**super().get_config(), 'horizon': self.horizon}

class StackHorizon(layers.Layer):
    """Stacks a 4D spatial tensor across the time dimension."""
    def __init__(self, horizon, **kwargs):
        super().__init__(**kwargs)
        self.horizon = horizon
    def call(self, x):
        return tf.stack([x] * self.horizon, axis=1)
    def get_config(self):
        return {**super().get_config(), 'horizon': self.horizon}

# ── 3. THE U-NET ARCHITECTURE ───────────────────────────────────────────
def build_cnn_lstm_unet(lookback=7, h=31, w=31, n_feats=5, horizon=3):
    inp = layers.Input(shape=(lookback, h, w, n_feats))

    # --- ENCODER (Spatial Feature Extraction) ---
    # Encoder — add kernel_regularizer to every Conv2D
    s1 = layers.TimeDistributed(
        layers.Conv2D(32, 3, padding='same', activation='relu',
                      kernel_regularizer=REG))(inp)
    s1 = layers.TimeDistributed(layers.BatchNormalization())(s1)
    s1 = layers.TimeDistributed(
        layers.Conv2D(64, 3, padding='same', activation='relu',
                      kernel_regularizer=REG))(s1)
    s1 = layers.TimeDistributed(layers.BatchNormalization())(s1)

    x = layers.TimeDistributed(layers.MaxPooling2D(2, padding='same'))(s1)

    s2 = layers.TimeDistributed(
        layers.Conv2D(128, 3, padding='same', activation='relu',
                      kernel_regularizer=REG))(x)
    s2 = layers.TimeDistributed(layers.BatchNormalization())(s2)

    # Bottleneck — increase Dropout from 0.3 to 0.4
    x = layers.ConvLSTM2D(128, kernel_size=3, padding='same',
                           return_sequences=False,recurrent_dropout=0.1)(s2)
    x = layers.Dropout(0.4)(x)   # was 0.3
    # x shape: (batch, 16, 16, 128)

    # 2. Duplicate that map into 3 future frames
    x = StackHorizon(horizon)(x)
    # x shape: (batch, 3, 16, 16, 128)

    # 3. Predict how the storm moves across those 3 frames
    x = layers.ConvLSTM2D(128, kernel_size=3, padding='same', return_sequences=True,recurrent_dropout=0.1)(x)
    # x shape: (batch, 3, 16, 16, 128)

    # --- DECODER (Spatial Reconstruction & Fusion) ---
    # Skip S2 fusion
    s2_mean = layers.Lambda(lambda t: tf.reduce_mean(t, axis=1))(s2) # Grab Day 7
    s2_exp  = TileHorizon(horizon)(s2_mean)                  # Expand to Day 8, 9, 10
    x       = layers.Concatenate(axis=-1)([x, s2_exp])
    # x shape: (batch, 3, 16, 16, 256)

    # Upsampling
    x  = layers.TimeDistributed(
        layers.Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu', kernel_regularizer=REG))(x)
    x  = layers.TimeDistributed(
        layers.Cropping2D(((0, 1), (0, 1))))(x)
    # x shape: (batch, 3, 31, 31, 64)

    # Skip S1 fusion
    s1_mean = layers.Lambda(lambda t: tf.reduce_mean(t, axis=1))(s1)
    s1_exp  = TileHorizon(horizon)(s1_mean)
    x       = layers.Concatenate(axis=-1)([x, s1_exp])
    # x shape: (batch, 3, 31, 31, 128)

    # Final Polish
    x  = layers.TimeDistributed(
        layers.Conv2D(32, 3, padding='same', activation='relu', kernel_regularizer=REG))(x)
    x  = layers.TimeDistributed(
        layers.BatchNormalization())(x)

    # Output Layer
    out = layers.TimeDistributed(
        layers.Conv2D(1, 1, activation='relu'))(x)
    out = layers.Reshape((horizon, h, w))(out)

    return Model(inp, out)

# ── 4. BUILD & COMPILE ──────────────────────────────────────────────────
print("Building Spatial-Temporal U-Net...")
model = build_cnn_lstm_unet()
model.summary()

Building Spatial-Temporal U-Net...



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 7, 31, 31, │          0 │ -                 │
│ (InputLayer)        │ 5)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 7, 31, 31, │      1,472 │ input_layer[0][0] │
│ (TimeDistributed)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1  │ (None, 7, 31, 31, │        128 │ time_distributed… │
│ (TimeDistributed)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_2  │ (None, 7, 31, 31, │     18,496 │ time_distributed… │
│ (TimeDistributed)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_3  │ (None, 7, 31, 31, │        256 │ time_distributed… │
│ (TimeDistributed)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_4  │ (None, 7, 16, 16, │          0 │ time_distributed… │
│ (TimeDistributed)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_5  │ (None, 7, 16, 16, │     73,856 │ time_distributed… │
│ (TimeDistributed)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_6  │ (None, 7, 16, 16, │        512 │ time_distributed… │
│ (TimeDistributed)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_lstm2d         │ (None, 16, 16,    │  1,180,160 │ time_distributed… │
│ (ConvLSTM2D)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 16, 16,    │          0 │ conv_lstm2d[0][0] │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_horizon       │ (None, 3, 16, 16, │          0 │ dropout[0][0]     │
│ (StackHorizon)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 16, 16,    │          0 │ time_distributed… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_lstm2d_1       │ (None, 3, 16, 16, │  1,180,160 │ stack_horizon[0]… │
│ (ConvLSTM2D)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tile_horizon        │ (None, 3, 16, 16, │          0 │ lambda[0][0]      │
│ (TileHorizon)       │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 3, 16, 16, │          0 │ conv_lstm2d_1[0]… │
│ (Concatenate)       │ 256)              │            │ tile_horizon[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_7  │ (None, 3, 32, 32, │    147,520 │ concatenate[0][0] │
│ (TimeDistributed)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 31, 31,    │          0 │ time_distributed

 Total params: 2,639,617 (10.07 MB)

 Trainable params: 2,639,105 (10.07 MB)

 Non-trainable params: 512 (2.00 KB)

In [3]:
# ══════════════════════════════════════════════════════════════
#  CELL 2 of prediction.ipynb — Load everything in one shot
#  Works in any fresh session, no recomputation needed
# ══════════════════════════════════════════════════════════════

import numpy as np
import pickle
from pathlib import Path

# Resolve pickle_files robustly across local Windows and Colab
candidate_dirs = [
    Path.cwd() / 'pickle_files',
    Path.cwd() / 'src' / 'Rainfall_ Model' / 'pickle_files',
    Path(r'C:/Users/ps302/OneDrive/Desktop/Hydrology/src/Rainfall_ Model/pickle_files'),
    Path('/content/drive/MyDrive/Rainfall_Prediction/pickle_files'),
]
SAVE_DIR = next((p for p in candidate_dirs if p.exists()), None)
if SAVE_DIR is None:
    raise FileNotFoundError('Could not find pickle_files. Checked: ' + ', '.join(str(p) for p in candidate_dirs))
print(f'Using SAVE_DIR: {SAVE_DIR}')

# 1. Load the full feature dataset (needed for lookback window)
print("Loading X_features from .npz (ERA5 weather)...")
dataset = np.load(SAVE_DIR / "Mahanadi_DeepLearning_Data.npz")
X_features = dataset['X']   # (7670, 31, 31, 5)
print(f"   X_features: {X_features.shape}")

# 2. Load y_clean (pre-processed IMD rainfall — no recomputation needed)
print("Loading y_clean from .npy (IMD rainfall)...")
y_clean = np.load(SAVE_DIR / 'y_clean.npy')   # (7670, 31, 31)
print(f"   y_clean: {y_clean.shape}")

# 3. Load the fitted scalers directly
print("Loading scalers...")
with open(SAVE_DIR / 'scaler_X.pkl', 'rb') as f:
    scaler_X = pickle.load(f)

with open(SAVE_DIR / 'scaler_y.pkl', 'rb') as f:
    scaler_y = pickle.load(f)

print("   scaler_X and scaler_y loaded.")

# 4. Training split reference (needed for WAY 3 climatology fallback)
TOTAL_DAYS = 7670
TRAIN_END  = int(TOTAL_DAYS * 0.70)  # 5369
X_train    = X_features[:TRAIN_END]

# 5. scale_array helper
def scale_array(arr, scaler, fit=False):
    if arr.ndim == 4:
        T, H, W, F = arr.shape
        flat = arr.reshape(-1, F)
        if fit:
            scaler.fit(flat)
        return scaler.transform(flat).reshape(T, H, W, F)
    else:
        T, H, W = arr.shape
        flat = arr.reshape(-1, 1)
        if fit:
            scaler.fit(flat)
        return scaler.transform(flat).reshape(T, H, W)

print()
print("All ready — X_features, y_clean, scaler_X, scaler_y, scale_array, X_train")
print("Run the prediction cell now.")

Using SAVE_DIR: c:\Users\ps302\OneDrive\Desktop\Hydrology\src\Rainfall_ Model\pickle_files
Loading X_features from .npz (ERA5 weather)...
   X_features: (7670, 31, 31, 5)
Loading y_clean from .npy (IMD rainfall)...
   y_clean: (7670, 31, 31)
Loading scalers...
   scaler_X and scaler_y loaded.

All ready — X_features, y_clean, scaler_X, scaler_y, scale_array, X_train
Run the prediction cell now.


Using SAVE_DIR: c:\Users\ps302\OneDrive\Desktop\Hydrology\src\Rainfall_ Model\pickle_files
Loading X_features from .npz (ERA5 weather)...
   X_features: (7670, 31, 31, 5)
Loading y_clean from .npy (IMD rainfall)...
   y_clean: (7670, 31, 31)
Loading scalers...
   scaler_X and scaler_y loaded.

All ready — X_features, y_clean, scaler_X, scaler_y, scale_array, X_train
Run the prediction cell now.


c:\Users\ps302\OneDrive\Desktop\Hydrology\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\ps302\OneDrive\Desktop\Hydrology\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [4]:
# Run this once per environment
from pathlib import Path

cdsapirc_path = Path.home() / '.cdsapirc'
cdsapirc_path.parent.mkdir(parents=True, exist_ok=True)

cdsapirc_content = """url: https://cds.climate.copernicus.eu/api/v2
key: 91c0ca32-abdf-4b66-8530-25a4f1155b4d"""

with open(cdsapirc_path, 'w', encoding='utf-8') as f:
    f.write(cdsapirc_content)

print(f'.cdsapirc created at {cdsapirc_path}')

.cdsapirc created at C:\Users\ps302\.cdsapirc


In [17]:
# ══════════════════════════════════════════════════════════════════════
#  SMART PREDICTION ENGINE — Final v5
#
#  WAY 1  — Date inside TRAIN range   (2005-01-01 → 2019-09-13)
#            Real ERA5 from X_features  | IMD actual  | Validation only
#
#  WAY 2  — Date inside VAL/TEST range (2019-09-14 → 2025-12-31)
#            Real ERA5 from X_features  | IMD actual  | Real blind test
#
#  WAY 3A — Beyond dataset BUT ≤ today
#            Live ERA5 fetched from Copernicus CDS  | No IMD ❌ | Operational
#
#  WAY 3B — Beyond today
#             BLOCKED — No real atmospheric data exists. Not a valid forecast.
# ══════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date

# ──   ONLY CHANGE THIS LINE ─────────────────────────────────────────
TARGET_START_DATE = '2020-04-06'
# ──────────────────────────────────────────────────────────────────────

MODEL_WEIGHTS    = r'C:\Users\ps302\OneDrive\Desktop\Hydrology\src\Rainfall_ Model\pickle_files\mahanadi_cnnlstm_unet11.keras'
DATASET_START    = pd.Timestamp('2005-01-01')
TODAY            = pd.Timestamp(date.today())
TOTAL_DAYS       = 7670
TRAIN_END        = int(TOTAL_DAYS * 0.70)
VAL_END          = int(TOTAL_DAYS * 0.85)
DATASET_END      = TOTAL_DAYS - 1
TRAIN_END_DATE   = DATASET_START + pd.Timedelta(days=TRAIN_END - 1)
DATASET_END_DATE = DATASET_START + pd.Timedelta(days=DATASET_END)
LOOKBACK         = 7
HORIZON          = 3
LAT_MIN, LAT_MAX = 17.0, 24.5
LON_MIN, LON_MAX = 80.0, 87.5

target_start     = pd.Timestamp(TARGET_START_DATE)
target_end       = target_start + pd.Timedelta(days=HORIZON - 1)
target_start_idx = (target_start - DATASET_START).days
target_end_idx   = (target_end   - DATASET_START).days

# Lookback ends ON Day 0 (includes current day ERA5)
lookback_start_idx = target_start_idx - (LOOKBACK - 1)
lookback_start     = DATASET_START + pd.Timedelta(days=lookback_start_idx)

# ── DETERMINE WAY ─────────────────────────────────────────────────────
target_in_dataset  = (target_start_idx >= 0) and (target_end_idx <= DATASET_END)
lookback_available = (lookback_start_idx >= 0) and \
                     (lookback_start_idx + LOOKBACK <= TOTAL_DAYS)

if target_start > TODAY:
    WAY = '3B'   # Future beyond today — BLOCKED
elif target_in_dataset and target_end_idx < TRAIN_END:
    WAY = 1      # Training range
elif target_in_dataset:
    WAY = 2      # Val/Test range
else:
    WAY = '3A'   # Beyond dataset but ≤ today — live ERA5 fetch

# ══════════════════════════════════════════════════════════════════════
#  WAY 3B — HARD BLOCK
#  Stop execution immediately. No prediction. No plot. Just a warning.
# ══════════════════════════════════════════════════════════════════════
if WAY == '3B':
    print("=" * 68)
    print("    PREDICTION BLOCKED — FUTURE DATE REQUESTED")
    print("=" * 68)
    print(f"""
  Target date   : {target_start.date()} → {target_end.date()}
  Today         : {TODAY.date()}
  Days ahead    : {(target_start - TODAY).days} days into the future

  WHY THIS IS BLOCKED:
  ─────────────────────────────────────────────────────────────
  Your model predicts rainfall using real ERA5 atmospheric data
  (temperature, pressure, humidity, wind) as input.

  For dates beyond today, this data does not exist anywhere —
  not in your dataset, not on Copernicus CDS, not anywhere.

  Feeding fake/averaged data would produce a number that looks
  like a forecast but has no meteorological meaning whatsoever.
  Showing it would be scientifically misleading.

  WHAT YOU CAN DO INSTEAD:
  ─────────────────────────────────────────────────────────────
    Use a date ≤ today ({TODAY.date()}) for a real prediction
      → WAY 3A will fetch live ERA5 from Copernicus CDS

    Use a date inside your dataset (≤ 2025-12-31)
      → WAY 1 or WAY 2 with stored ERA5 data

    For genuine future forecasting beyond 3 days,
      use NWP model output (ECMWF/GFS) as input instead
      of ERA5 — that is a separate, more complex pipeline.

  SCIENTIFIC LIMIT OF RAINFALL FORECASTING:
  ─────────────────────────────────────────────────────────────
  Day 1–3  →  Valid  (your model's designed range) 
  Day 4–7  →  Uncertain (needs ensemble methods)   ⚠️
  Day 8+   →  Not reliable for any model            ❌
  Future   →  Impossible without NWP input          ❌
""")
    print("=" * 68)
    print("  Change TARGET_START_DATE to a date ≤ today and re-run.")
    print("=" * 68)

    # Stop here — do not run any code below
    raise SystemExit("Prediction blocked: future date has no real atmospheric input.")


# ══════════════════════════════════════════════════════════════════════
#  WAYS 1, 2, 3A — Normal prediction flow
# ══════════════════════════════════════════════════════════════════════

WAY_DESCRIPTIONS = {
    1  : "🟢 WAY 1  — Training range   | real ERA5  | actual IMD  | validation only",
    2  : "🟡 WAY 2  — Val/Test range   | real ERA5  | actual IMD  | genuine blind test",
    '3A': "🟠 WAY 3A — Beyond dataset, ≤ today | live ERA5 from CDS  | operational use",
}

print(f"{'='*68}")
print(f"  Model   : {MODEL_WEIGHTS.split('/')[-1]}")
print(f"  Today   : {TODAY.date()}")
print(f"  Target  : {target_start.date()} → {target_end.date()}")
print(f"  Lookback: {lookback_start.date()} → {target_start.date()}")
print(f"{'='*68}")
print(f"  {WAY_DESCRIPTIONS[WAY]}")
print(f"{'='*68}\n")


# ── ERA5 LIVE FETCH HELPER (WAY 3A only) ──────────────────────────────
def fetch_era5_for_window(start_date, n_days):
    """
    Fetches n_days of ERA5 data from Copernicus CDS.
    Returns (n_days, 31, 31, 5) — same format as X_features.
    Channel order: [temp, pressure, humidity, u_wind, v_wind]

    Prerequisites:
      pip install cdsapi xarray netCDF4
      ~/.cdsapirc with your CDS UID and API key
    """
    import cdsapi, xarray as xr, tempfile, os

    dates  = [start_date + pd.Timedelta(days=i) for i in range(n_days)]
    years  = list(set(str(d.year)        for d in dates))
    months = list(set(f"{d.month:02d}"   for d in dates))
    days_l = list(set(f"{d.day:02d}"     for d in dates))

    c = cdsapi.Client()

    surface_vars  = ['2m_temperature', 'mean_sea_level_pressure',
                     '10m_u_component_of_wind', '10m_v_component_of_wind']
    humidity_vars = ['specific_humidity']

    with tempfile.TemporaryDirectory() as tmpdir:

        # Surface: t2m, msl, u10, v10
        surface_path = os.path.join(tmpdir, 'surface.nc')
        c.retrieve('reanalysis-era5-single-levels', {
            'product_type': 'reanalysis',
            'variable'    : surface_vars,
            'year'        : years,
            'month'       : months,
            'day'         : days_l,
            'time'        : '12:00',
            'area'        : [24.5, 80.0, 17.0, 87.5],   # N, W, S, E
            'format'      : 'netcdf',
        }, surface_path)

        # Humidity: q at 850 hPa (separate product — pressure levels)
        humidity_path = os.path.join(tmpdir, 'humidity.nc')
        c.retrieve('reanalysis-era5-pressure-levels', {
            'product_type'  : 'reanalysis',
            'variable'      : humidity_vars,
            'pressure_level': '850',
            'year'          : years,
            'month'         : months,
            'day'           : days_l,
            'time'          : '12:00',
            'area'          : [24.5, 80.0, 17.0, 87.5],
            'format'        : 'netcdf',
        }, humidity_path)

        # Interpolate to training grid (31x31)
        LATS = np.linspace(17.0, 24.5, 31)
        LONS = np.linspace(80.0, 87.5, 31)

        ds_s = xr.open_dataset(surface_path).sortby('latitude').sortby('longitude')
        ds_h = xr.open_dataset(humidity_path).sortby('latitude').sortby('longitude')

        ds_s = ds_s.interp(latitude=LATS, longitude=LONS)
        ds_h = ds_h.interp(latitude=LATS, longitude=LONS)

        date_strs = [str(d.date()) for d in dates]
        ds_s = ds_s.sel(time=date_strs)
        ds_h = ds_h.sel(time=date_strs)

        temp     = ds_s['t2m'].values
        pressure = ds_s['msl'].values
        u_wind   = ds_s['u10'].values
        v_wind   = ds_s['v10'].values
        humidity = ds_h['q'].squeeze().values

        # ⚠️ Channel order MUST match training: [temp, pressure, humidity, u_wind, v_wind]
        X_live = np.stack([temp, pressure, humidity, u_wind, v_wind], axis=-1)

    print(f" ERA5 fetched: {date_strs[0]} → {date_strs[-1]}, shape: {X_live.shape}")
    return X_live


# ── STEP 1: LOAD MODEL ────────────────────────────────────────────────
model = build_cnn_lstm_unet()
model.load_weights(MODEL_WEIGHTS)
print(" Model weights loaded.")

# ── STEP 2: BUILD LOOKBACK INPUT ──────────────────────────────────────
if WAY in (1, 2):
    X_window_raw    = X_features[lookback_start_idx : lookback_start_idx + LOOKBACK]
    X_window_scaled = scale_array(X_window_raw, scaler_X, fit=False)
    print(f" Lookback from X_features: {lookback_start.date()} → {target_start.date()}")

else:  # WAY 3A
    print("🟠 WAY 3A: Fetching live ERA5 from Copernicus CDS...")
    print(f"   Requesting: {lookback_start.date()} → {target_start.date()} ({LOOKBACK} days)")
    try:
        X_window_raw    = fetch_era5_for_window(lookback_start, LOOKBACK)
        X_window_scaled = scale_array(X_window_raw, scaler_X, fit=False)
        print(" Live ERA5 fetched and scaled.")
    except Exception as e:
        print(f"\ CDS fetch failed: {e}")
        print("   Check your ~/.cdsapirc credentials and internet connection.")
        raise

X_input = X_window_scaled[np.newaxis, ...]    # (1, 7, 31, 31, 5)

# ── STEP 3: PREDICT ───────────────────────────────────────────────────
predicted_scaled = model.predict(X_input, verbose=0)
predicted_scaled = np.clip(predicted_scaled, 0, None)

def unscale(arr, scaler):
    return scaler.inverse_transform(arr.reshape(-1, 1)).reshape(arr.shape)

pred_mm = unscale(predicted_scaled[0], scaler_y)   # (3, 31, 31)
print(" Prediction complete.")

# ── STEP 4: ACTUAL IMD RAINFALL (WAY 1 & 2 only) ─────────────────────
has_actual = False
if WAY in (1, 2):
    actual_raw = y_clean[target_start_idx : target_start_idx + HORIZON]
    if actual_raw.shape[0] == HORIZON:
        actual_mm  = actual_raw
        has_actual = True
        print(f" Actual IMD data: {target_start.date()} → {target_end.date()}")

# ── STEP 5: PLOT ──────────────────────────────────────────────────────
n_rows     = 2 if has_actual else 1
row_labels = ['ACTUAL (IMD)', 'PREDICTED'] if has_actual else ['PREDICTED']
data_rows  = [actual_mm, pred_mm]          if has_actual else [pred_mm]

way_label = {
    1  : "Way 1 — Training data (validation)",
    2  : "Way 2 — Blind Val/Test (real evaluation)",
    '3A': "Way 3A — Live ERA5 fetch (operational forecast)",
}

fig, axes = plt.subplots(n_rows, HORIZON,
                          figsize=(6 * HORIZON, 5 * n_rows),
                          squeeze=False)
fig.suptitle(
    f"Mahanadi Basin: 3-Day Rainfall Forecast\n"
    f"{target_start.date()} → {target_end.date()}   |   {way_label[WAY]}",
    fontsize=13, fontweight='bold'
)

for row_i, (label, maps) in enumerate(zip(row_labels, data_rows)):
    for d in range(HORIZON):
        ax       = axes[row_i, d]
        day_date = target_start + pd.Timedelta(days=d)
        vmax     = max(
            float(actual_mm[d].max()) if has_actual else 0.0,
            float(pred_mm[d].max()), 1.0
        )
        im = ax.imshow(maps[d], cmap='YlGnBu', vmin=0, vmax=vmax,
                       origin='lower', extent=[LON_MIN, LON_MAX, LAT_MIN, LAT_MAX])
        ax.set_title(f"{label}\n{day_date.strftime('%d %b %Y')}", fontsize=11)
        ax.set_xlabel("Longitude (°E)", fontsize=9)
        ax.set_ylabel("Latitude (°N)",  fontsize=9)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='mm')
        if has_actual and label == 'PREDICTED':
            mae = float(np.mean(np.abs(actual_mm[d] - pred_mm[d])))
            ax.set_xlabel(f"Day {d+1} MAE = {mae:.2f} mm", fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

# ── STEP 6: SUMMARY ───────────────────────────────────────────────────
print("\n📊 Basin-average results:")
print(f"  {'Date':<14} {'Pred mean':>10} {'Pred max':>10}"
      + (f" {'Actual mean':>12} {'Day MAE':>10}" if has_actual else ""))
print("  " + "─" * (46 + (24 if has_actual else 0)))

for d in range(HORIZON):
    day_date = target_start + pd.Timedelta(days=d)
    pm       = float(np.mean(pred_mm[d]))
    px       = float(np.max(pred_mm[d]))
    line     = f"  {str(day_date.date()):<14} {pm:>10.2f} {px:>10.2f}"
    if has_actual:
        am   = float(np.mean(actual_mm[d]))
        mae  = float(np.mean(np.abs(actual_mm[d] - pred_mm[d])))
        line += f" {am:>12.2f} {mae:>10.2f}"
    print(line)

if has_actual:
    print(f"\n  Overall 3-day MAE: {float(np.mean(np.abs(actual_mm - pred_mm))):.2f} mm")

if WAY == '3A':
    print(f"\n  ℹ️  WAY 3A: Prediction used live ERA5 data fetched from Copernicus CDS.")
    print(f"     No IMD actual available — this is a real operational forecast.")

<>:234: SyntaxWarning: invalid escape sequence '\ '
<>:234: SyntaxWarning: invalid escape sequence '\ '
C:\Users\ps302\AppData\Local\Temp\ipykernel_15908\3989673610.py:234: SyntaxWarning: invalid escape sequence '\ '
  print(f"\ CDS fetch failed: {e}")


  Model   : C:\Users\ps302\OneDrive\Desktop\Hydrology\src\Rainfall_ Model\pickle_files\mahanadi_cnnlstm_unet11.keras
  Today   : 2026-04-06
  Target  : 2020-04-06 → 2020-04-08
  Lookback: 2020-03-31 → 2020-04-06
  🟡 WAY 2  — Val/Test range   | real ERA5  | actual IMD  | genuine blind test



ValueError: A total of 2 objects could not be loaded. Example error message for object <Conv2D name=conv2d_54, built=True>:

'Unable to synchronously open object (message not aligned)'

List of objects that could not be loaded:
[<Conv2D name=conv2d_54, built=True>, <Reshape name=reshape_10, built=True>]

In [13]:
def predict_at_location(lat, lon):
    """
    Given a lat/lon:
    1. Finds the nearest grid point
    2. Selects all pixels within ±1 degree of that nearest point
    3. For each of the 3 forecast days, finds the pixel with MAX predicted
       rainfall inside that rectangle and reports its coordinates,
       along with max pred, mean pred, and max actual at that same pixel.
    """

    LATS = np.linspace(17.0, 24.5, 31)
    LONS = np.linspace(80.0, 87.5, 31)

    # ── Step 1: nearest grid point ───────────────────────────────
    nearest_row = int(np.argmin(np.abs(LATS - lat)))
    nearest_col = int(np.argmin(np.abs(LONS - lon)))
    nearest_lat = LATS[nearest_row]
    nearest_lon = LONS[nearest_col]

    # ── Step 2: all row/col indices within ±1 degree ─────────────
    row_indices = np.where(np.abs(LATS - nearest_lat) <= 1.0)[0]
    col_indices = np.where(np.abs(LONS - nearest_lon) <= 1.0)[0]

    # Sub-grid slice boundaries (for efficient indexing)
    r_min, r_max = row_indices[0],  row_indices[-1]
    c_min, c_max = col_indices[0],  col_indices[-1]

    n_pixels = len(row_indices) * len(col_indices)

    # ── Step 3: Print header ─────────────────────────────────────
    print(f"Query     : {lat:.2f}°N, {lon:.2f}°E")
    print(f"Nearest   : {nearest_lat:.2f}°N, {nearest_lon:.2f}°E")
    print(f"Area      : {LATS[r_min]:.2f}–{LATS[r_max]:.2f}°N, "
          f"{LONS[c_min]:.2f}–{LONS[c_max]:.2f}°E  ({n_pixels} pixels)")
    print()

    if has_actual:
        print(f"  {'Date':<14} {'Max coord':>18} {'Max pred':>10} "
              f"{'Mean pred':>10} {'Actual @ max':>13}")
        print(f"  {'─'*70}")
    else:
        print(f"  {'Date':<14} {'Max coord':>18} {'Max pred':>10} {'Mean pred':>10}")
        print(f"  {'─'*57}")

    # ── Step 4: Per-day analysis ─────────────────────────────────
    for d in range(HORIZON):
        day_date = target_start + pd.Timedelta(days=d)

        # Extract the sub-grid patch for predicted rainfall
        pred_patch = pred_mm[d, r_min:r_max+1, c_min:c_max+1]
        # shape: (n_rows_in_area, n_cols_in_area)

        # Find position of max predicted value within the patch
        flat_idx        = int(np.argmax(pred_patch))
        local_row, local_col = np.unravel_index(flat_idx, pred_patch.shape)

        # Convert back to global grid indices
        global_row = r_min + local_row
        global_col = c_min + local_col

        # Real-world coordinates of the max pixel
        max_lat = LATS[global_row]
        max_lon = LONS[global_col]

        max_pred  = float(pred_patch[local_row, local_col])
        mean_pred = float(pred_patch.mean())

        coord_str = f"({max_lat:.2f}°N,{max_lon:.2f}°E)"

        if has_actual:
            # Actual value at the SAME pixel that had max predicted rainfall
            actual_at_max = float(actual_mm[d, global_row, global_col])
            print(f"  {str(day_date.date()):<14} {coord_str:>18} {max_pred:>10.2f} "
                  f"{mean_pred:>10.2f} {actual_at_max:>12.2f}")
        else:
            print(f"  {str(day_date.date()):<14} {coord_str:>18} {max_pred:>10.2f} "
                  f"{mean_pred:>10.2f}")

    print()


# ── Call it ──────────────────────────────────────────────────────
predict_at_location(20.5, 83.5)
predict_at_location(22.0, 82.0)
predict_at_location(19.0, 85.0)

Query     : 20.50°N, 83.50°E
Nearest   : 20.50°N, 83.50°E
Area      : 19.50–21.50°N, 82.50–84.50°E  (81 pixels)

  Date                    Max coord   Max pred  Mean pred  Actual @ max
  ──────────────────────────────────────────────────────────────────────
  2020-04-06      (19.50°N,82.50°E)       0.00       0.00         0.00
  2020-04-07      (19.50°N,83.00°E)       1.32       0.09         0.00
  2020-04-08      (19.50°N,83.00°E)       1.18       0.11         0.00

Query     : 22.00°N, 82.00°E
Nearest   : 22.00°N, 82.00°E
Area      : 21.00–23.00°N, 81.00–83.00°E  (81 pixels)

  Date                    Max coord   Max pred  Mean pred  Actual @ max
  ──────────────────────────────────────────────────────────────────────
  2020-04-06      (21.00°N,81.00°E)       0.00       0.00         0.00
  2020-04-07      (21.00°N,81.00°E)       0.00       0.00         0.00
  2020-04-08      (21.00°N,81.00°E)       0.00       0.00         0.00

Query     : 19.00°N, 85.00°E
Nearest   : 19.00°N, 85.00°